# Tetrasphere demo

Loads the shipped bac120 genome-stage checkpoint (fold 1) and embeds two real representative genomes, to show the embedding geometry: direction encodes relative phylogenetic position, norm encodes root-to-tip depth.

In [ ]:
import logging
import gzip
import numpy as np
import torch
from Bio import SeqIO

from config import config
from src.model import checkpoint
from src.tetrasphere.model import Manifold
from src.tetrasphere.kmer import Kmer

logger = logging.getLogger("demo")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
ckpt_path = checkpoint.find(config.result_checkpoint / "bac120" / "bac120_4mer_clr_genome" / "fold_1", None)
model_config = checkpoint.load_config(ckpt_path)["model_config"]
model = Manifold(**model_config).to(device)
checkpoint.load_weights(ckpt_path, model, device)
model.eval()
print(model_config)

In [ ]:
# Two real GTDB representative genomes: Escherichia coli and Bacillus subtilis
GENOMES = {
    "Escherichia coli": "GCF_003697165.2",
    "Bacillus subtilis": "GCF_000009045.1",
}

def load_sequence(accession: str) -> str:
    path = config.gtdb_download / "representative_genomes" / f"{accession}_genomic.fna.gz"
    with gzip.open(path, "rt") as f:
        return "".join(str(record.seq) for record in SeqIO.parse(f, "fasta"))

kmer = Kmer(k=4, logger=logger)
sequences = {name: load_sequence(acc) for name, acc in GENOMES.items()}
features = np.stack([kmer.vectorize(seq, transform_kmer="clr") for seq in sequences.values()])
for name, seq in sequences.items():
    print(f"{name}: {len(seq):,} bp")

In [ ]:
with torch.no_grad():
    embeddings = model.encode(torch.from_numpy(features).float().to(device)).cpu().numpy()

z_a, z_b = embeddings
cos_sim = float(np.dot(z_a, z_b) / (np.linalg.norm(z_a) * np.linalg.norm(z_b)))
print(f"cosine similarity: {cos_sim:.4f}")
print(f"embedding norms: {np.linalg.norm(z_a):.4f}, {np.linalg.norm(z_b):.4f}")

Lower cosine similarity means a larger embedding-space angle between the two genomes, consistent with their phylogenetic distance; each embedding's norm tracks that genome's own root-to-tip depth in the reference tree.

See `scripts/figures/fig2b_kf2vec_comparison.py` for how these quantities are converted into an actual distance/depth estimate and scored against ground truth, and `UNIT_TEST.md` for the full pipeline.